# 🍅 ポモドーロタイマー

下のセルを実行するとタイマーが表示されます。

**機能一覧:**
- ⏱ 作業(25分) / 短い休憩(5分) / 長い休憩(15分) の自動切替
- ⚙️ 時間設定のカスタマイズ
- 📋 タスク管理（追加・完了・削除）
- 📊 統計表示（今日のポモドーロ数・累計）
- 🔔 通知音
- 🌙 ダークモード対応

In [2]:
from IPython.display import display, HTML

html_code = """
<div id="pomodoro-app">
<style>
  @import url('https://fonts.googleapis.com/css2?family=Inter:wght@300;400;500;600;700&family=JetBrains+Mono:wght@400;700&display=swap');

  :root {
    --bg: #F5F5F5;
    --fg: #333333;
    --accent: #E74C3C;
    --accent-break: #27AE60;
    --accent-long-break: #2980B9;
    --btn-bg: #FFFFFF;
    --btn-fg: #333333;
    --btn-hover: #E8E8E8;
    --panel-bg: #FFFFFF;
    --arc-bg: #E0E0E0;
    --input-bg: #FFFFFF;
    --input-border: #D0D0D0;
    --label-fg: #888888;
    --shadow: 0 2px 12px rgba(0,0,0,0.08);
    --task-done: #AAAAAA;
  }

  [data-theme=\"dark\"] {
    --bg: #1E1E2E;
    --fg: #CDD6F4;
    --accent: #F38BA8;
    --accent-break: #A6E3A1;
    --accent-long-break: #89B4FA;
    --btn-bg: #313244;
    --btn-fg: #CDD6F4;
    --btn-hover: #45475A;
    --panel-bg: #181825;
    --arc-bg: #313244;
    --input-bg: #313244;
    --input-border: #45475A;
    --label-fg: #A6ADC8;
    --shadow: 0 2px 12px rgba(0,0,0,0.3);
    --task-done: #6C7086;
  }

  #pomodoro-app {
    font-family: 'Inter', sans-serif;
    background: var(--bg);
    color: var(--fg);
    max-width: 520px;
    margin: 0 auto;
    padding: 24px;
    border-radius: 16px;
    box-shadow: var(--shadow);
    transition: all 0.3s ease;
  }

  #pomodoro-app * { box-sizing: border-box; }

  .pm-header {
    text-align: center;
    margin-bottom: 8px;
  }
  .pm-header h2 {
    margin: 0;
    font-size: 22px;
    font-weight: 700;
    color: var(--fg);
    transition: color 0.3s;
  }
  .pm-mode-label {
    font-size: 14px;
    color: var(--label-fg);
    margin-top: 4px;
    transition: color 0.3s;
  }

  .pm-timer-section {
    display: flex;
    flex-direction: column;
    align-items: center;
    margin: 16px 0;
  }

  .pm-canvas-wrap {
    position: relative;
    width: 220px;
    height: 220px;
  }

  .pm-canvas-wrap canvas {
    width: 220px;
    height: 220px;
  }

  .pm-time-display {
    position: absolute;
    top: 50%;
    left: 50%;
    transform: translate(-50%, -55%);
    font-family: 'JetBrains Mono', monospace;
    font-size: 42px;
    font-weight: 700;
    color: var(--fg);
    transition: color 0.3s;
  }
  .pm-mode-sub {
    position: absolute;
    top: 62%;
    left: 50%;
    transform: translateX(-50%);
    font-size: 13px;
    color: var(--label-fg);
    transition: color 0.3s;
  }

  .pm-controls {
    display: flex;
    gap: 8px;
    justify-content: center;
    margin: 12px 0;
    flex-wrap: wrap;
  }

  .pm-btn {
    padding: 8px 18px;
    border: none;
    border-radius: 8px;
    background: var(--btn-bg);
    color: var(--btn-fg);
    font-family: 'Inter', sans-serif;
    font-size: 13px;
    font-weight: 500;
    cursor: pointer;
    transition: all 0.2s;
    box-shadow: 0 1px 4px rgba(0,0,0,0.06);
  }
  .pm-btn:hover { background: var(--btn-hover); }
  .pm-btn:disabled { opacity: 0.4; cursor: default; }
  .pm-btn.primary {
    background: var(--accent);
    color: #fff;
  }
  .pm-btn.primary:hover { filter: brightness(1.1); }
  .pm-btn-skip {
    background: transparent;
    border: none;
    color: var(--label-fg);
    font-size: 12px;
    cursor: pointer;
    padding: 4px 8px;
    font-family: 'Inter', sans-serif;
  }
  .pm-btn-skip:hover { color: var(--fg); }

  .pm-panels {
    display: grid;
    grid-template-columns: 1fr 1fr;
    gap: 12px;
    margin-top: 16px;
  }

  .pm-panel {
    background: var(--panel-bg);
    border-radius: 10px;
    padding: 14px;
    box-shadow: 0 1px 4px rgba(0,0,0,0.04);
    transition: background 0.3s;
  }

  .pm-panel-title {
    font-size: 13px;
    font-weight: 600;
    margin-bottom: 10px;
    color: var(--fg);
  }

  /* タスクパネル */
  .pm-task-input-row {
    display: flex;
    gap: 6px;
    margin-bottom: 8px;
  }
  .pm-task-input {
    flex: 1;
    padding: 6px 10px;
    border: 1px solid var(--input-border);
    border-radius: 6px;
    background: var(--input-bg);
    color: var(--fg);
    font-family: 'Inter', sans-serif;
    font-size: 12px;
    outline: none;
    transition: all 0.2s;
  }
  .pm-task-input:focus { border-color: var(--accent); }
  .pm-task-add-btn {
    padding: 6px 12px;
    border: none;
    border-radius: 6px;
    background: var(--accent);
    color: #fff;
    font-size: 14px;
    font-weight: 700;
    cursor: pointer;
  }
  .pm-task-add-btn:hover { filter: brightness(1.1); }

  .pm-task-list {
    list-style: none;
    padding: 0;
    margin: 0;
    max-height: 140px;
    overflow-y: auto;
  }
  .pm-task-item {
    display: flex;
    align-items: center;
    gap: 6px;
    padding: 5px 4px;
    border-radius: 4px;
    font-size: 12px;
    transition: background 0.15s;
  }
  .pm-task-item:hover { background: var(--btn-hover); }
  .pm-task-item.done { color: var(--task-done); text-decoration: line-through; }
  .pm-task-item input[type=checkbox] {
    accent-color: var(--accent);
    cursor: pointer;
  }
  .pm-task-item .pm-task-del {
    margin-left: auto;
    background: none;
    border: none;
    color: var(--label-fg);
    cursor: pointer;
    font-size: 14px;
    padding: 0 4px;
    line-height: 1;
  }
  .pm-task-item .pm-task-del:hover { color: var(--accent); }

  /* 統計パネル */
  .pm-stat-row {
    display: flex;
    justify-content: space-between;
    padding: 4px 0;
    font-size: 12px;
    border-bottom: 1px solid var(--arc-bg);
  }
  .pm-stat-row:last-child { border-bottom: none; }
  .pm-stat-label { color: var(--label-fg); }
  .pm-stat-value { font-weight: 600; color: var(--accent); }

  /* 設定パネル */
  .pm-setting-row {
    display: flex;
    justify-content: space-between;
    align-items: center;
    padding: 3px 0;
    font-size: 12px;
  }
  .pm-setting-row label { color: var(--label-fg); }
  .pm-setting-input {
    width: 50px;
    padding: 4px 6px;
    border: 1px solid var(--input-border);
    border-radius: 4px;
    background: var(--input-bg);
    color: var(--fg);
    font-family: 'JetBrains Mono', monospace;
    font-size: 12px;
    text-align: center;
    outline: none;
  }
  .pm-setting-input:focus { border-color: var(--accent); }

  .pm-dark-toggle {
    display: flex;
    align-items: center;
    gap: 8px;
    margin-top: 8px;
    font-size: 12px;
    color: var(--label-fg);
    cursor: pointer;
  }

  /* switch */
  .pm-switch {
    position: relative;
    width: 36px;
    height: 20px;
  }
  .pm-switch input { opacity: 0; width: 0; height: 0; }
  .pm-switch .slider {
    position: absolute;
    top: 0; left: 0; right: 0; bottom: 0;
    background: var(--arc-bg);
    border-radius: 20px;
    cursor: pointer;
    transition: 0.3s;
  }
  .pm-switch .slider:before {
    content: '';
    position: absolute;
    width: 14px; height: 14px;
    left: 3px; bottom: 3px;
    background: #fff;
    border-radius: 50%;
    transition: 0.3s;
  }
  .pm-switch input:checked + .slider { background: var(--accent); }
  .pm-switch input:checked + .slider:before { transform: translateX(16px); }

  .pm-full-width { grid-column: 1 / -1; }
</style>

<div class=\"pm-header\">
  <h2>🍅 ポモドーロタイマー</h2>
  <div class=\"pm-mode-label\" id=\"pmCycleLabel\">サイクル: 1 / 4</div>
</div>

<div class=\"pm-timer-section\">
  <div class=\"pm-canvas-wrap\">
    <canvas id=\"pmCanvas\" width=\"440\" height=\"440\"></canvas>
    <div class=\"pm-time-display\" id=\"pmTimeDisplay\">25:00</div>
    <div class=\"pm-mode-sub\" id=\"pmModeSub\">作業中</div>
  </div>
</div>

<div class=\"pm-controls\">
  <button class=\"pm-btn primary\" id=\"pmStartBtn\" onclick=\"pmStart()\">▶ 開始</button>
  <button class=\"pm-btn\" id=\"pmPauseBtn\" onclick=\"pmPause()\" disabled>⏸ 一時停止</button>
  <button class=\"pm-btn\" id=\"pmResetBtn\" onclick=\"pmReset()\">⏹ リセット</button>
</div>
<div style=\"text-align:center;\">
  <button class=\"pm-btn-skip\" onclick=\"pmSkip()\">次へスキップ ⏭</button>
</div>

<div class=\"pm-panels\">
  <!-- タスク -->
  <div class=\"pm-panel\">
    <div class=\"pm-panel-title\">📋 タスク</div>
    <div class=\"pm-task-input-row\">
      <input type=\"text\" class=\"pm-task-input\" id=\"pmTaskInput\" placeholder=\"タスクを追加...\" onkeydown=\"if(event.key==='Enter')pmAddTask()\">
      <button class=\"pm-task-add-btn\" onclick=\"pmAddTask()\">+</button>
    </div>
    <ul class=\"pm-task-list\" id=\"pmTaskList\"></ul>
  </div>

  <!-- 統計 -->
  <div class=\"pm-panel\">
    <div class=\"pm-panel-title\">📊 統計</div>
    <div class=\"pm-stat-row\"><span class=\"pm-stat-label\">今日</span><span class=\"pm-stat-value\" id=\"pmStatToday\">0 ポモドーロ</span></div>
    <div class=\"pm-stat-row\"><span class=\"pm-stat-label\">作業時間</span><span class=\"pm-stat-value\" id=\"pmStatTime\">0 分</span></div>
    <div class=\"pm-stat-row\"><span class=\"pm-stat-label\">累計</span><span class=\"pm-stat-value\" id=\"pmStatTotal\">0 ポモドーロ</span></div>
  </div>

  <!-- 設定 -->
  <div class=\"pm-panel pm-full-width\">
    <div class=\"pm-panel-title\">⚙️ 設定</div>
    <div style=\"display:grid;grid-template-columns:1fr 1fr;gap:4px 16px;\">
      <div class=\"pm-setting-row\"><label>作業(分)</label><input class=\"pm-setting-input\" id=\"pmSetWork\" type=\"number\" value=\"25\" min=\"1\"></div>
      <div class=\"pm-setting-row\"><label>短い休憩(分)</label><input class=\"pm-setting-input\" id=\"pmSetShort\" type=\"number\" value=\"5\" min=\"1\"></div>
      <div class=\"pm-setting-row\"><label>長い休憩(分)</label><input class=\"pm-setting-input\" id=\"pmSetLong\" type=\"number\" value=\"15\" min=\"1\"></div>
      <div class=\"pm-setting-row\"><label>サイクル数</label><input class=\"pm-setting-input\" id=\"pmSetCycles\" type=\"number\" value=\"4\" min=\"1\"></div>
    </div>
    <div style=\"display:flex;gap:12px;align-items:center;margin-top:10px;\">
      <button class=\"pm-btn\" onclick=\"pmApplySettings()\" style=\"font-size:12px;padding:6px 14px;\">適用</button>
      <label class=\"pm-dark-toggle\">
        <span class=\"pm-switch\"><input type=\"checkbox\" id=\"pmDarkToggle\" onchange=\"pmToggleDark()\"><span class=\"slider\"></span></span>
        🌙 ダークモード
      </label>
    </div>
  </div>
</div>
</div>

<script>
(function() {
  // --- State ---
  const S = {
    workMin: 25, shortBreakMin: 5, longBreakMin: 15, cyclesBeforeLong: 4,
    mode: 'work',   // 'work' | 'short_break' | 'long_break'
    running: false,
    timeLeft: 25 * 60,
    totalTime: 25 * 60,
    completedCycles: 0,
    timerId: null,
    tasks: [],
    stats: {},     // { 'YYYY-MM-DD': { pomodoros, minutes } }
    dark: false
  };

  const MODES = { work: '作業中', short_break: '短い休憩', long_break: '長い休憩' };

  // --- Persistence (localStorage) ---
  function saveState() {
    try {
      localStorage.setItem('pm_tasks', JSON.stringify(S.tasks));
      localStorage.setItem('pm_stats', JSON.stringify(S.stats));
      localStorage.setItem('pm_settings', JSON.stringify({
        workMin: S.workMin, shortBreakMin: S.shortBreakMin,
        longBreakMin: S.longBreakMin, cyclesBeforeLong: S.cyclesBeforeLong, dark: S.dark
      }));
    } catch(e) {}
  }

  function loadState() {
    try {
      const t = localStorage.getItem('pm_tasks');
      if (t) S.tasks = JSON.parse(t);
      const st = localStorage.getItem('pm_stats');
      if (st) S.stats = JSON.parse(st);
      const se = localStorage.getItem('pm_settings');
      if (se) {
        const o = JSON.parse(se);
        S.workMin = o.workMin || 25;
        S.shortBreakMin = o.shortBreakMin || 5;
        S.longBreakMin = o.longBreakMin || 15;
        S.cyclesBeforeLong = o.cyclesBeforeLong || 4;
        S.dark = !!o.dark;
      }
    } catch(e) {}
    S.timeLeft = S.workMin * 60;
    S.totalTime = S.timeLeft;
  }

  // --- DOM refs ---
  const $ = id => document.getElementById(id);
  const app = $('pomodoro-app');
  const canvas = $('pmCanvas');
  const ctx = canvas.getContext('2d');

  // --- Drawing ---
  function getAccentColor() {
    if (S.mode === 'work') return S.dark ? '#F38BA8' : '#E74C3C';
    if (S.mode === 'short_break') return S.dark ? '#A6E3A1' : '#27AE60';
    return S.dark ? '#89B4FA' : '#2980B9';
  }

  function drawProgress() {
    const w = canvas.width, h = canvas.height;
    const cx = w / 2, cy = h / 2, r = 180, lw = 18;
    ctx.clearRect(0, 0, w, h);

    // bg arc
    ctx.beginPath();
    ctx.arc(cx, cy, r, 0, Math.PI * 2);
    ctx.strokeStyle = S.dark ? '#313244' : '#E0E0E0';
    ctx.lineWidth = lw;
    ctx.lineCap = 'round';
    ctx.stroke();

    // progress arc
    const progress = S.totalTime > 0 ? S.timeLeft / S.totalTime : 0;
    if (progress > 0.001) {
      const startAngle = -Math.PI / 2;
      const endAngle = startAngle + Math.PI * 2 * progress;
      ctx.beginPath();
      ctx.arc(cx, cy, r, startAngle, endAngle);
      ctx.strokeStyle = getAccentColor();
      ctx.lineWidth = lw;
      ctx.lineCap = 'round';
      ctx.stroke();
    }
  }

  function updateDisplay() {
    const m = Math.floor(S.timeLeft / 60);
    const s = S.timeLeft % 60;
    const ts = String(m).padStart(2, '0') + ':' + String(s).padStart(2, '0');
    $('pmTimeDisplay').textContent = ts;
    $('pmModeSub').textContent = MODES[S.mode];
    $('pmCycleLabel').textContent = 'サイクル: ' + (S.completedCycles % S.cyclesBeforeLong + 1) + ' / ' + S.cyclesBeforeLong;
    drawProgress();
    // update accent CSS variable for stat-value
    app.style.setProperty('--accent', getAccentColor());
  }

  function updateStats() {
    const today = new Date().toISOString().slice(0, 10);
    const td = S.stats[today] || { pomodoros: 0, minutes: 0 };
    let totalP = 0, totalM = 0;
    for (const k in S.stats) { totalP += S.stats[k].pomodoros; totalM += S.stats[k].minutes; }
    $('pmStatToday').textContent = td.pomodoros + ' ポモドーロ';
    $('pmStatTime').textContent = td.minutes + ' 分';
    $('pmStatTotal').textContent = totalP + ' ポモドーロ (' + totalM + '分)';
  }

  // --- Timer logic ---
  function tick() {
    if (!S.running) return;
    if (S.timeLeft > 0) {
      S.timeLeft--;
      updateDisplay();
      S.timerId = setTimeout(tick, 1000);
    } else {
      onTimerEnd();
    }
  }

  function playNotification() {
    try {
      const actx = new (window.AudioContext || window.webkitAudioContext)();
      const osc = actx.createOscillator();
      const gain = actx.createGain();
      osc.connect(gain); gain.connect(actx.destination);
      osc.type = 'sine'; osc.frequency.value = 800;
      gain.gain.value = 0.3;
      osc.start(); osc.stop(actx.currentTime + 0.5);
      setTimeout(() => {
        const osc2 = actx.createOscillator();
        const gain2 = actx.createGain();
        osc2.connect(gain2); gain2.connect(actx.destination);
        osc2.type = 'sine'; osc2.frequency.value = 1000;
        gain2.gain.value = 0.3;
        osc2.start(); osc2.stop(actx.currentTime + 0.5);
      }, 600);
    } catch(e) {}
  }

  function onTimerEnd() {
    playNotification();
    if (S.mode === 'work') {
      const today = new Date().toISOString().slice(0, 10);
      if (!S.stats[today]) S.stats[today] = { pomodoros: 0, minutes: 0 };
      S.stats[today].pomodoros++;
      S.stats[today].minutes += S.workMin;
      S.completedCycles++;
      updateStats();
      saveState();
    }
    switchMode();
    updateDisplay();
    S.timerId = setTimeout(tick, 1000);
  }

  function switchMode() {
    if (S.mode === 'work') {
      if (S.completedCycles > 0 && S.completedCycles % S.cyclesBeforeLong === 0) {
        S.mode = 'long_break';
        S.timeLeft = S.longBreakMin * 60;
      } else {
        S.mode = 'short_break';
        S.timeLeft = S.shortBreakMin * 60;
      }
    } else {
      S.mode = 'work';
      S.timeLeft = S.workMin * 60;
    }
    S.totalTime = S.timeLeft;
  }

  // --- Global functions (called from HTML) ---
  window.pmStart = function() {
    if (S.running) return;
    S.running = true;
    $('pmStartBtn').disabled = true;
    $('pmPauseBtn').disabled = false;
    tick();
  };

  window.pmPause = function() {
    S.running = false;
    if (S.timerId) { clearTimeout(S.timerId); S.timerId = null; }
    $('pmStartBtn').disabled = false;
    $('pmPauseBtn').disabled = true;
  };

  window.pmReset = function() {
    S.running = false;
    if (S.timerId) { clearTimeout(S.timerId); S.timerId = null; }
    S.mode = 'work';
    S.timeLeft = S.workMin * 60;
    S.totalTime = S.timeLeft;
    S.completedCycles = 0;
    $('pmStartBtn').disabled = false;
    $('pmPauseBtn').disabled = true;
    updateDisplay();
  };

  window.pmSkip = function() {
    const wasRunning = S.running;
    S.running = false;
    if (S.timerId) { clearTimeout(S.timerId); S.timerId = null; }
    switchMode();
    updateDisplay();
    if (wasRunning) { window.pmStart(); }
    else {
      $('pmStartBtn').disabled = false;
      $('pmPauseBtn').disabled = true;
    }
  };

  // --- Tasks ---
  function renderTasks() {
    const ul = $('pmTaskList');
    ul.innerHTML = '';
    S.tasks.forEach((t, i) => {
      const li = document.createElement('li');
      li.className = 'pm-task-item' + (t.done ? ' done' : '');
      li.innerHTML = '<input type=\"checkbox\"' + (t.done ? ' checked' : '') + '>' +
        '<span>' + t.text.replace(/</g,'&lt;') + '</span>' +
        '<button class=\"pm-task-del\">✕</button>';
      li.querySelector('input').onchange = function() {
        S.tasks[i].done = this.checked;
        saveState(); renderTasks();
      };
      li.querySelector('.pm-task-del').onclick = function() {
        S.tasks.splice(i, 1);
        saveState(); renderTasks();
      };
      ul.appendChild(li);
    });
  }

  window.pmAddTask = function() {
    const inp = $('pmTaskInput');
    const text = inp.value.trim();
    if (!text) return;
    S.tasks.push({ text: text, done: false });
    inp.value = '';
    saveState(); renderTasks();
  };

  // --- Settings ---
  window.pmApplySettings = function() {
    const w = parseInt($('pmSetWork').value) || 25;
    const sb = parseInt($('pmSetShort').value) || 5;
    const lb = parseInt($('pmSetLong').value) || 15;
    const c = parseInt($('pmSetCycles').value) || 4;
    S.workMin = Math.max(1, w);
    S.shortBreakMin = Math.max(1, sb);
    S.longBreakMin = Math.max(1, lb);
    S.cyclesBeforeLong = Math.max(1, c);
    saveState();
    if (!S.running) window.pmReset();
  };

  window.pmToggleDark = function() {
    S.dark = $('pmDarkToggle').checked;
    app.setAttribute('data-theme', S.dark ? 'dark' : 'light');
    saveState();
    updateDisplay();
  };

  // --- Init ---
  loadState();
  // apply loaded settings to inputs
  $('pmSetWork').value = S.workMin;
  $('pmSetShort').value = S.shortBreakMin;
  $('pmSetLong').value = S.longBreakMin;
  $('pmSetCycles').value = S.cyclesBeforeLong;
  $('pmDarkToggle').checked = S.dark;
  if (S.dark) app.setAttribute('data-theme', 'dark');
  renderTasks();
  updateStats();
  updateDisplay();
})();
</script>
"""

display(HTML(html_code))